In [37]:
from sklearn.datasets import fetch_openml

In [38]:
import pandas as pd

In [39]:
X,y  = fetch_openml("titanic",version=1,as_frame=True,return_X_y=True)

In [40]:
print("Feature matrix shape",X.shape)

Feature matrix shape (1309, 13)


In [41]:
print("Target shape:",y.shape)

Target shape: (1309,)


In [42]:
X.head()



,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [43]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   name       1309 non-null   object  
 2   sex        1309 non-null   category
 3   age        1046 non-null   float64 
 4   sibsp      1309 non-null   int64   
 5   parch      1309 non-null   int64   
 6   ticket     1309 non-null   object  
 7   fare       1308 non-null   float64 
 8   cabin      295 non-null    object  
 9   embarked   1307 non-null   category
 10  boat       486 non-null    object  
 11  body       121 non-null    float64 
 12  home.dest  745 non-null    object  
dtypes: category(2), float64(3), int64(3), object(5)
memory usage: 115.4+ KB


In [44]:
y.head()


,survived
0,1
1,1
2,0
3,0
4,0


In [45]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1309 entries, 0 to 1308
Series name: survived
Non-Null Count  Dtype   
--------------  -----   
1309 non-null   category
dtypes: category(1)
memory usage: 1.5 KB


In [46]:
#count and sort missing values
missing = X.isna().sum().sort_values(ascending=False)
print(missing[missing>0])

body         1188
cabin        1014
boat          823
home.dest     564
age           263
embarked        2
fare            1
dtype: int64


In [47]:

print(X["age"].describe())


count    1046.000000
mean       29.881135
std        14.413500
min         0.166700
25%        21.000000
50%        28.000000
75%        39.000000
max        80.000000
Name: age, dtype: float64


In [48]:
print(X["sex"].value_counts())

sex
male      843
female    466
Name: count, dtype: int64


In [49]:
numeric_features=["age","fare"]
categorical_features=["sex","embarked","pclass"]
#subset our feature matrix ro just these columns
X_titanic = X[numeric_features + categorical_features].copy()
X_titanic.head()


,age,fare,sex,embarked,pclass
0,29.0000,211.3375,female,S,1
1,0.9167,151.5500,male,S,1
2,2.0000,151.5500,female,S,1
3,30.0000,151.5500,male,S,1
4,25.0000,151.5500,female,S,1


In [50]:
print("Missing values in selected features:")
print(X_titanic.isna().sum())

Missing values in selected features:
age         263
fare          1
sex           0
embarked      2
pclass        0
dtype: int64


In [51]:
from sklearn.model_selection import train_test_split
# step 4 is train test split before any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X_titanic,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,   # preserve class balance

)
print("Train:",X_train.shape,"Test:",X_test.shape)


Train: (1047, 5) Test: (262, 5)


In [52]:
#check missing values in the training dataset only
X_train.isna().sum()




,0
age,209
fare,1
sex,0
embarked,0
pclass,0


In [53]:
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train[["age"]])

print("Learn median from age:", num_imputer.statistics_)

# transform both train and test
age_train_imputed = num_imputer.transform(X_train[["age"]])
age_test_imputed = num_imputer.transform(X_test[["age"]])

print("\nMissing in age after imputation(train):", pd.isnull(age_train_imputed).sum())
print("Missing in age after imputation(test):", pd.isnull(age_test_imputed).sum())

Learn median from age: [28.]

Missing in age after imputation(train): 0
Missing in age after imputation(test): 0


In [54]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
#numeric pipeline: impute then scale
numeric_transformer= Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])
# categorical pipeline : fill missing then one-hot encode
categorical_transformer =Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="constant",fill_value="missing")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))

])

In [55]:
#quick test on numeric transformer
test_age= X_train[["age"]].head(10)
print("Before:\n",test_age.to_string())
result = numeric_transformer.fit_transform(test_age)
print("\nAfter(imputed + scaled):\n",result)

Before:
        age
999    NaN
392   24.0
628   11.0
1165  25.0
604   16.0
605   25.0
25    25.0
1278  20.0
196    NaN
1292   NaN

After(imputed + scaled):
 [[ 0.48756643]
 [ 0.48756643]
 [-2.39350792]
 [ 0.70918753]
 [-1.2854024 ]
 [ 0.70918753]
 [ 0.70918753]
 [-0.39891799]
 [ 0.48756643]
 [ 0.48756643]]
